# Day 32: Tool Use & Function Calling

**Week 5 — Agent Development**

---

## 📚 Theory: How Function Calling Works

"Function Calling" is the mechanism by which an LLM interacts with the outside world. 
Crucially, **the LLM does not execute the function**. 

The lifecycle is:
1. You pass a JSON schema of tools to the LLM.
2. The LLM decides it needs to use a tool, so it responds with a JSON string indicating the `tool_name` and `arguments` instead of a text response.
3. **Your backend code** parses this JSON, finds the local Python function, executes it with the provided arguments, and gets a result.
4. You append the result as a new "Tool Response" message and send it *back* to the LLM so it can continue reasoning.

### Safely Executing Tools
Because LLMs can hallucinate, they might pass invalid arguments (e.g., passing a string when an integer is required) or call tools that don't exist.
Your ADK must gracefully catch these errors and return them to the LLM. Often, an LLM is smart enough to read the Python traceback and fix its own mistake on the next iteration!

## 🔨 Exercise 1: The Tool Executor

Let's build a safe tool executor. 

**Task**: Write a function `execute_tool(tool_call, available_tools)` that takes a mock JSON response from an LLM and safely executes the corresponding Python function. If the function fails or doesn't exist, it should return a gracefully formatted error string.

In [ ]:
def add_numbers(a: int, b: int) -> int:
    return a + b

def divide_numbers(a: int, b: int) -> float:
    return a / b

my_tools = {
    "add_numbers": add_numbers,
    "divide_numbers": divide_numbers
}

def execute_tool(tool_call: dict, available_tools: dict) -> str:
    """
    tool_call format: {"name": "add_numbers", "args": {"a": 5, "b": 10}}
    Returns the stringified result, or an error message.
    """
    # YOUR CODE HERE
    pass

# Test 1: Valid Call
assert execute_tool({"name": "add_numbers", "args": {"a": 5, "b": 10}}, my_tools) == "15"

# Test 2: Invalid Tool Name
# print(execute_tool({"name": "fake_tool", "args": {}}, my_tools))

# Test 3: Division by zero
# print(execute_tool({"name": "divide_numbers", "args": {"a": 10, "b": 0}}, my_tools))

In [ ]:
# Solution
import traceback

def execute_tool_solution(tool_call: dict, available_tools: dict) -> str:
    tool_name = tool_call.get("name")
    args = tool_call.get("args", {})
    
    if tool_name not in available_tools:
        return f"Error: Tool '{tool_name}' not found. Available tools: {list(available_tools.keys())}"
        
    func = available_tools[tool_name]
    
    try:
        # Execute the function using kwargs unpacking
        result = func(**args)
        return str(result)
    except Exception as e:
        # Capture the error gracefully so the LLM can see what went wrong
        error_msg = f"Error executing {tool_name}: {str(e)}"
        return error_msg

print("Solution Outputs:")
print(execute_tool_solution({"name": "add_numbers", "args": {"a": 5, "b": 10}}, my_tools))
print(execute_tool_solution({"name": "fake_tool", "args": {}}, my_tools))
print(execute_tool_solution({"name": "divide_numbers", "args": {"a": 10, "b": 0}}, my_tools))


## 🔨 Exercise 2: RAG (Retrieval-Augmented Generation) as a Tool

The most common tool an agent uses is a **RAG Retriever**. LLMs only know facts up to their training cutoff date. To answer questions about your private company docs or current events, they must use a Retrieval Tool.

**Task**: Create a mock Vector Database retrieval tool. The tool will take a query, compute its "similarity" against a database of documents, and return the most relevant document.

In [ ]:
mock_vector_db = [
    {"doc_id": 1, "text": "Google was founded in 1998 by Larry Page and Sergey Brin.", "tags": ["company", "history"]},
    {"doc_id": 2, "text": "Gemini 1.5 Pro features a 2 million token context window.", "tags": ["ai", "llm"]},
    {"doc_id": 3, "text": "The capital of France is Paris.", "tags": ["geography"]}
]

def search_knowledge_base(query_tag: str) -> str:
    """
    Searches the mock_vector_db for a document containing the exact query_tag in its tags list.
    Returns the text of the document, or 'No relevant documents found.'
    """
    # YOUR CODE HERE
    pass

assert search_knowledge_base("ai") == "Gemini 1.5 Pro features a 2 million token context window."
assert search_knowledge_base("sports") == "No relevant documents found."
print("All tests passed! ✅")

In [ ]:
# Solution
def search_knowledge_base_solution(query_tag: str) -> str:
    for doc in mock_vector_db:
        if query_tag in doc["tags"]:
            return doc["text"]
            
    return "No relevant documents found."


## 📝 Day 32 Review

### Concepts Validated Today
- [ ] The distinction between an LLM *choosing* a tool and your backend *executing* the tool.
- [ ] Wrapping tool execution in `try/except` blocks to feed errors back into the ReAct loop.
- [ ] How **Retrieval-Augmented Generation (RAG)** is simply just a specific type of Search Tool provided to the agent.

### Tomorrow's Preview
**Day 33: Testing and Debugging Agents** — How do you unit test an application where the output is non-deterministic? We will cover LLM-as-a-Judge and evasion testing.